# Адаптерный триминг + фильтрация — человек (`PRJEB40348`)

Схема: `cutadapt` снимает только технические 5′ key/MID префиксы и Illumina/NEBNext адаптеры read-through; `fastp` затем фильтрует целые пары (`-q 30 -u 40 -l 250`) без обрезки концов по качеству и без повторной автодетекции адаптеров. Результат атомарно заменяет `results/PRJEB40348/trimmed`. Контроль качества выполняется отдельно.


## Адаптеры для этого датасета

| Что | Статус | Значение |
|---|---|---|
| 5′ A-key префикс | есть, восстановлен эмпирически | `CGTATCGCCTCCCTCGCGCCATCAGACGCCTCGAGGCGGCCGCTCTAGA` |
| 5′ B-key+MID префиксы (6 вариантов) | есть, восстановлены эмпирически | список `B_KEYS` |
| Illumina/NEBNext R1 read-through | есть, подтверждён эмпирически | `AGATCGGAAGAGCACACGTCTGAACTCCAGTCA` |
| Illumina/NEBNext R2 read-through | есть, подтверждён эмпирически | `AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT` |
| Биологические VH/JH праймеры | НЕ режутся здесь | отдельная стадия `primer_trim_human.ipynb` |

Источник: набор указан (NEBNext Ultra + NEBNext Multiplex Oligos), точные олигонуклеотиды адаптеров в публикации не приведены — последовательности подтверждены эмпирически (fastp JSON + проверки cutadapt), семейство TruSeq/NEBNext, не Nextera.

In [ ]:
import os, sys, sysconfig, shutil, subprocess, time, json
from pathlib import Path
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [_CONDA_ENV + "/lib/python3.11/site-packages", sysconfig.get_path("purelib")]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)
print("cutadapt:", shutil.which("cutadapt"))
print("fastp:", shutil.which("fastp"))


In [ ]:
# --- фильтрация (единая схема fastp_q30_u40) ---
QUALITY_PHRED = 30
UNQUALIFIED_PERCENT_LIMIT = 40
MIN_LENGTH = 250

# --- одинаковые адаптеры для всех запусков ---
ADAPTER_TIMES = 2
ADAPTER_MIN_OVERLAP = 10
ADAPTERS_BY_RUN = None  # None: указанные ниже константы применяются ко всем запускам.

A_KEY = 'CGTATCGCCTCCCTCGCGCCATCAGACGCCTCGAGGCGGCCGCTCTAGA'
B_KEYS = [
    'CTATGCGCCTTGCCAGCCCGCTCAGACGAGTGCGTACTTGGCTAGCGCCAAGCTTGCTGA',
    'CTATGCGCCTTGCCAGCCCGCTCAGACGCTCGACAACTTGGCTAGCGCCAAGCTTGCTGA',
    'CTATGCGCCTTGCCAGCCCGCTCAGAGACGCACTCACTTGGCTAGCGCCAAGCTTGCTGA',
    'CTATGCGCCTTGCCAGCCCGCTCAGAGCACTGTAGACTTGGCTAGCGCCAAGCTTGCTGA',
    'CTATGCGCCTTGCCAGCCCGCTCAGATCAGACACGACTTGGCTAGCGCCAAGCTTGCTGA',
    'CTATGCGCCTTGCCAGCCCGCTCAGATATCGCGAGACTTGGCTAGCGCCAAGCTTGCTGA',
]
ADAPTER_PREFIXES = [A_KEY] + B_KEYS
ILLUMINA_ADAPTER_R1 = 'AGATCGGAAGAGCACACGTCTGAACTCCAGTCA'
ILLUMINA_ADAPTER_R2 = 'AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT'
ADAPTER_R1 = ILLUMINA_ADAPTER_R1
ADAPTER_R2 = ILLUMINA_ADAPTER_R2
DATASET = 'PRJEB40348'


In [ ]:

def _tool(name):
    path = shutil.which(name)
    if not path:
        raise FileNotFoundError(name)
    return path

def _run_visible(cmd, stdout_log, stderr_log, outputs=(), heartbeat=30):
    started = time.monotonic()
    print('[run]', ' '.join(map(str, cmd)), flush=True)
    with open(stdout_log, 'w') as stdout, open(stderr_log, 'w') as stderr:
        proc = subprocess.Popen([str(x) for x in cmd], stdout=stdout, stderr=stderr, text=True)
        print(f'PID={proc.pid}', flush=True)
        while proc.poll() is None:
            sizes = ' '.join(f'{Path(x).name}={Path(x).stat().st_size / 1e6:.1f}MB'
                             for x in outputs if Path(x).exists())
            print(f'PID={proc.pid} elapsed={(time.monotonic()-started)/60:.1f}min {sizes}', flush=True)
            time.sleep(heartbeat)
    if proc.returncode:
        raise RuntimeError(f'rc={proc.returncode}; see {stderr_log}')

def _promote_stage(staging, final):
    previous = final.parent / f'.{final.name}.previous'
    if previous.exists():
        shutil.rmtree(previous)
    if final.exists():
        final.rename(previous)
    try:
        staging.rename(final)
    except Exception:
        if previous.exists() and not final.exists():
            previous.rename(final)
        raise
    if previous.exists():
        shutil.rmtree(previous)

def adapters_for_run(run_name):
    """Возвращает (prefixes, r1, r2). Пустой кортеж/None = адаптеров для этого run нет,
    и тогда cutadapt не запускается — сразу fastp-фильтрация."""
    if ADAPTERS_BY_RUN is not None:
        return ADAPTERS_BY_RUN.get(run_name, ((), None, None))
    return (ADAPTER_PREFIXES, ADAPTER_R1, ADAPTER_R2)

def run_adapter_trim(volume, dataset=DATASET, force=False):
    vol = Path(volume)
    src = vol / 'raw' / dataset
    final = vol / 'results' / dataset / 'trimmed'
    staging = final.parent / '.trimmed_fastp_q30_u40.staging'
    if not src.is_dir():
        raise FileNotFoundError(src)
    pairs = sorted(p.name.removesuffix('_1.fastq.gz') for p in src.glob('*_1.fastq.gz'))
    if not pairs or any(not (src / f'{bn}_2.fastq.gz').is_file() for bn in pairs):
        raise RuntimeError(f'Incomplete paired FASTQ set in {src}')
    if staging.exists():
        if not force:
            raise FileExistsError(f'Stale staging exists: {staging}; rerun with force=True')
        shutil.rmtree(staging)
    out = staging / 'fastq'
    logs = staging / 'logs'
    reports = staging / 'fastp_reports'
    tmp = staging / 'adapter_only_tmp'
    for d in (out, logs, reports, tmp):
        d.mkdir(parents=True, exist_ok=True)
    print(f'[adapter_filter] {dataset}: {len(pairs)} pairs; Q{QUALITY_PHRED}/u{UNQUALIFIED_PERCENT_LIMIT}/min{MIN_LENGTH}', flush=True)
    summaries, no_adapter = {}, []
    for bn in pairs:
        r1 = src / f'{bn}_1.fastq.gz'
        r2 = src / f'{bn}_2.fastq.gz'
        a1 = tmp / f'{bn}_1.adapter.fastq.gz'
        a2 = tmp / f'{bn}_2.adapter.fastq.gz'
        o1 = out / f'{bn}_1.trim.fastq.gz'
        o2 = out / f'{bn}_2.trim.fastq.gz'
        prefixes, adapter_r1, adapter_r2 = adapters_for_run(bn)
        adapters_r1 = [x for x in (adapter_r1 if isinstance(adapter_r1, (list, tuple)) else [adapter_r1]) if x]
        adapters_r2 = [x for x in (adapter_r2 if isinstance(adapter_r2, (list, tuple)) else [adapter_r2]) if x]
        if prefixes or adapters_r1 or adapters_r2:
            ca = [_tool('cutadapt'), '--times', str(ADAPTER_TIMES), '-O', str(ADAPTER_MIN_OVERLAP),
                  '--compression-level', '1']
            for prefix in prefixes:
                ca += ['-g', '^' + prefix, '-G', '^' + prefix]
            for adapter in adapters_r1:
                ca += ['-a', adapter]
            for adapter in adapters_r2:
                ca += ['-A', adapter]
            ca += ['--json', logs / f'{bn}.cutadapt.json', '-o', a1, '-p', a2, r1, r2]
            _run_visible(ca, logs / f'{bn}.cutadapt.stdout.log', logs / f'{bn}.cutadapt.stderr.log', [a1, a2])
            fastp_in_1, fastp_in_2 = a1, a2
        else:
            print(f'[{bn}] адаптеров не найдено — cutadapt пропущен, сразу fastp-фильтрация', flush=True)
            no_adapter.append(bn)
            fastp_in_1, fastp_in_2 = r1, r2
        jf = reports / f'{bn}.fastp.json'
        hf = reports / f'{bn}.fastp.html'
        fp = [_tool('fastp'), '-i', fastp_in_1, '-I', fastp_in_2, '-o', o1, '-O', o2,
              '-q', str(QUALITY_PHRED), '-u', str(UNQUALIFIED_PERCENT_LIMIT), '-l', str(MIN_LENGTH),
              '--disable_adapter_trimming', '--disable_trim_poly_g', '-w', '4', '-j', jf, '-h', hf]
        _run_visible(fp, logs / f'{bn}.fastp.stdout.log', logs / f'{bn}.fastp.stderr.log', [o1, o2])
        for path in (a1, a2):
            if path.exists():
                path.unlink()
        d = json.loads(jf.read_text())
        s = d['summary']
        fr = d['filtering_result']
        summaries[bn] = {
            'adapters_used': bool(prefixes or adapters_r1 or adapters_r2),
            'before_pairs': s['before_filtering']['total_reads'] // 2,
            'after_pairs': s['after_filtering']['total_reads'] // 2,
            'retention': round(s['after_filtering']['total_reads'] / s['before_filtering']['total_reads'], 4),
            'low_quality_reads': fr['low_quality_reads'],
            'too_short_reads': fr['too_short_reads'],
        }
    if len(list(out.glob('*.trim.fastq.gz'))) != 2 * len(pairs) or len(list(reports.glob('*.fastp.json'))) != len(pairs):
        raise RuntimeError('Output completeness validation failed; canonical trimmed stage was not replaced')
    summary = {
        'quality_semantics': 'cutadapt (технические префиксы и/или адаптеры) + fastp whole-read filter; без обрезки концов по качеству',
        'qualified_quality_phred': QUALITY_PHRED,
        'unqualified_percent_limit': UNQUALIFIED_PERCENT_LIMIT,
        'minimum_length': MIN_LENGTH,
        'runs_without_adapters': no_adapter,
        'pairs': summaries,
    }
    (staging / 'filter_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False) + '\n')
    shutil.rmtree(tmp)
    _promote_stage(staging, final)
    print(f'[adapter_filter] DONE and promoted: {final}', flush=True)


## Запуск с атомарной заменой стадии
Каталог `trimmed` заменяется только после успешной обработки и проверки всех пар.


In [ ]:
run_adapter_trim('/data/user/epishkin', 'PRJEB40348', force=True)
